# 05 — Tables and Summary

Synthesise manuscript Tables 1–2, generate manifests, capture supplementary report.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "src"))

OUT_FIGURES = REPO_ROOT / "outputs" / "figures"
OUT_TABLES  = REPO_ROOT / "outputs" / "tables"
OUT_DATA    = REPO_ROOT / "outputs" / "data"
OUT_LOGS    = REPO_ROOT / "outputs" / "logs"
for d in [OUT_FIGURES, OUT_TABLES, OUT_DATA, OUT_LOGS]:
    d.mkdir(parents=True, exist_ok=True)

import json, hashlib, shutil, io, contextlib
import pandas as pd

In [ ]:
# Load computed outputs
with open(OUT_DATA / "metrics_summary.json") as f:
    primary = json.load(f)
verif = pd.read_csv(OUT_DATA / "verification_summary.csv")
noise_ext = pd.read_csv(OUT_DATA / "noise_extended.csv")

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""): h.update(chunk)
    return h.hexdigest()

def get_verif(scenario, method):
    row = verif[(verif["Scenario"]==scenario) & (verif["Decision Rule"]==method)]
    if len(row) == 0: return "N/A"
    return f"{row.iloc[0]['Unsafe Deployment Rate']*100:.1f}%"


In [ ]:
# Table 1: Scope-condition summary
t1 = [
    {"Evidence Distribution": "Heterogeneous (primary model; simulated)",
     "Gate Safety Advantage vs Composite (Moderate)": "Strong",
     "Mechanism": "Compensation mechanism exploitable; conjunctive rule prevents it structurally"},
    {"Evidence Distribution": "Partial heterogeneity (verification simulation)",
     "Gate Safety Advantage vs Composite (Moderate)": "Moderate (confirmed)",
     "Mechanism": "Gates 0.8% vs composite 7.4%; partial compensation exploitable"},
    {"Evidence Distribution": "Random failure (verification simulation)",
     "Gate Safety Advantage vs Composite (Moderate)": "Small-moderate (confirmed)",
     "Mechanism": "Gates 0.9% vs composite 6.1%; random combinations include heterogeneous patterns"},
    {"Evidence Distribution": "Uniform failure (verification simulation)",
     "Gate Safety Advantage vs Composite (Moderate)": "None (confirmed)",
     "Mechanism": "No compensation mechanism; both architectures equivalent"},
    {"Evidence Distribution": "Noise sensitivity (simulated)",
     "Gate Safety Advantage vs Composite (Moderate)": "Robust",
     "Mechanism": "Gate safety maintained across noise range at matched deployment rates"},
]
pd.DataFrame(t1).to_csv(OUT_TABLES / "table1_scope_conditions.csv", index=False)
print("Table 1 saved.")

In [ ]:
# Table 2: Unsafe deployment rates
r0 = noise_ext.iloc[0]; r1 = noise_ext.iloc[1]
t2 = [
    {"Evidence Distribution Scenario": "Heterogeneous (primary model)",
     "Non-Comp. Gates": "0.0%",
     "Composite (Moderate)": f"{primary['metrics']['Composite (mean)']['unsafe_deployment_rate']*100:.1f}%-{primary['metrics']['Composite (mean)']['unsafe_among_deployed']*100:.1f}%",
     "Composite (Matched)": "0.0%",
     "Permissive Baseline": f"{primary['metrics']['Permissive']['unsafe_deployment_rate']*100:.1f}%"},
    {"Evidence Distribution Scenario": "Uniform failure (verified)",
     "Non-Comp. Gates": get_verif("Uniform Failure","Gates"), "Composite (Moderate)": get_verif("Uniform Failure","Composite (moderate)"),
     "Composite (Matched)": get_verif("Uniform Failure","Composite (matched)"), "Permissive Baseline": get_verif("Uniform Failure","Permissive")},
    {"Evidence Distribution Scenario": "Random failure (verified)",
     "Non-Comp. Gates": get_verif("Random Failure","Gates"), "Composite (Moderate)": get_verif("Random Failure","Composite (moderate)"),
     "Composite (Matched)": get_verif("Random Failure","Composite (matched)"), "Permissive Baseline": get_verif("Random Failure","Permissive")},
    {"Evidence Distribution Scenario": "Partial heterogeneity (verified)",
     "Non-Comp. Gates": get_verif("Partial Heterogeneity","Gates"), "Composite (Moderate)": get_verif("Partial Heterogeneity","Composite (moderate)"),
     "Composite (Matched)": get_verif("Partial Heterogeneity","Composite (matched)"), "Permissive Baseline": get_verif("Partial Heterogeneity","Permissive")},
    {"Evidence Distribution Scenario": "Noise robustness: Low (SD=0.01)",
     "Non-Comp. Gates": f"{r0['gate_unsafe_rate']*100:.1f}%",
     "Composite (Moderate)": f"{r0['composite_matched_unsafe_rate']*100:.1f}% (matched)",
     "Composite (Matched)": f"{r0['composite_matched_unsafe_rate']*100:.1f}%",
     "Permissive Baseline": f"{r0['permissive_unsafe_rate']*100:.1f}%"},
    {"Evidence Distribution Scenario": "Noise robustness: High (SD=0.20)",
     "Non-Comp. Gates": f"{r1['gate_unsafe_rate']*100:.1f}%",
     "Composite (Moderate)": f"{r1['composite_matched_unsafe_rate']*100:.1f}% (matched)",
     "Composite (Matched)": f"{r1['composite_matched_unsafe_rate']*100:.1f}%",
     "Permissive Baseline": f"{r1['permissive_unsafe_rate']*100:.1f}%"},
]
pd.DataFrame(t2).to_csv(OUT_TABLES / "table2_unsafe_rates.csv", index=False)
print("Table 2 saved.")

In [ ]:
# Copy static Figure 1
shutil.copy2(REPO_ROOT / "data" / "static" / "Figure1_DecisionRuleComparison.png",
             OUT_FIGURES / "Figure1_DecisionRuleComparison.png")
print("Static Figure 1 copied.")

In [ ]:
# Generate manifests
fig_lines = ["Paper 2 Figure Manifest"]
for f in sorted(OUT_FIGURES.glob("*.png")):
    fig_lines.append(f"{f.name}  {sha256_file(f)}")
(OUT_FIGURES / "paper2_figure_manifest.txt").write_text("\n".join(fig_lines) + "\n", encoding="utf-8")

tbl_lines = ["file,sha256"]
for f in sorted(OUT_TABLES.glob("*.csv")):
    tbl_lines.append(f"{f.name},{sha256_file(f)}")
(OUT_TABLES / "paper2_table_manifest.csv").write_text("\n".join(tbl_lines) + "\n", encoding="utf-8")
print("Manifests generated.")

In [ ]:
# Supplementary report
sys.path.insert(0, str(REPO_ROOT / "src"))
buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    from report_supplementary import main as report_main
    report_main(["--output-dir", str(OUT_DATA)])
(OUT_LOGS / "supplementary_report.txt").write_text(buf.getvalue(), encoding="utf-8")
print("Supplementary report captured.")

## Known Annotations\n\n- **A-01:** Permissive at noise SD=0.01 computes 2.0% vs manuscript 1.9% (1-tool boundary difference)\n- **A-02:** Threshold multipliers 0.60–0.65 show 0.1–0.2% non-zero gate leakage; manuscript rounds to zero\n- **A-03:** Epic case NaN RuntimeWarning is cosmetic\n- **A-04:** Figure hashes are platform-dependent (ADVISORY policy)